In [0]:
import tensorflow as tf
import numpy as np
import pandas as pd

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
data = pd.read_csv('/content/drive/My Drive/ANN/bank.csv')

In [56]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [57]:
data.shape

(10000, 14)

In [58]:
data.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [59]:
data.isnull().any()

RowNumber          False
CustomerId         False
Surname            False
CreditScore        False
Geography          False
Gender             False
Age                False
Tenure             False
Balance            False
NumOfProducts      False
HasCrCard          False
IsActiveMember     False
EstimatedSalary    False
Exited             False
dtype: bool

In [60]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
RowNumber          10000 non-null int64
CustomerId         10000 non-null int64
Surname            10000 non-null object
CreditScore        10000 non-null int64
Geography          10000 non-null object
Gender             10000 non-null object
Age                10000 non-null int64
Tenure             10000 non-null int64
Balance            10000 non-null float64
NumOfProducts      10000 non-null int64
HasCrCard          10000 non-null int64
IsActiveMember     10000 non-null int64
EstimatedSalary    10000 non-null float64
Exited             10000 non-null int64
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [0]:
#Dropping RowNumber, CustomerID and Surname as they will not have any impact on churn
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [62]:
data.shape

(10000, 11)

In [63]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [0]:
#Separating features and target
X = data.drop('Exited',axis=1)
y = data['Exited']

In [65]:
#Checking unique values of Geography and Gender. 
print ("Unique Geography values:", X['Geography'].unique())
print ("Unique Gender values   :", X['Gender'].unique())

Unique Geography values: ['France' 'Spain' 'Germany']
Unique Gender values   : ['Female' 'Male']


In [0]:
#Geography and Gender are of type Object. Using get_dummies on Geography and Gender. 
#Once the new columns are added, We will not need Geography and Gender columns. So i will drop them.

In [0]:
X_Geo = pd.get_dummies(X['Geography'],drop_first=True)

In [0]:
X_Gen = pd.get_dummies(X['Gender'],drop_first=True)

In [0]:
X = pd.concat([X_Geo, X], axis = 1)

In [0]:
X = pd.concat([X_Gen, X], axis = 1)

In [71]:
X.head()

,Male,Germany,Spain,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,0,0,0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,0,0,1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,0,0,0,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,0,0,0,699,France,Female,39,1,0.00,2,0,0,93826.63
4,0,0,1,850,Spain,Female,43,2,125510.82,1,1,1,79084.10


In [0]:
#Dropping Geography and Gender columns
X = X.drop(['Geography','Gender'],axis=1)

In [73]:
X.head()

,Male,Germany,Spain,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,0,0,0,619,42,2,0.00,1,1,1,101348.88
1,0,0,1,608,41,1,83807.86,1,0,1,112542.58
2,0,0,0,502,42,8,159660.80,3,1,0,113931.57
3,0,0,0,699,39,1,0.00,2,0,0,93826.63
4,0,0,1,850,43,2,125510.82,1,1,1,79084.10


In [0]:
#Splitting data in train and test

In [0]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

In [0]:
sc=StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [77]:
X_train.shape

(8000, 11)

In [0]:
import keras
from keras.models import Sequential
from keras.layers import Dense

In [82]:
model = Sequential()
model.add(Dense(output_dim = 6, init = 'uniform', activation = 'relu', input_dim = 11))
model.add(Dense(output_dim = 6, init = 'uniform', activation = 'relu'))
model.add(Dense(output_dim = 1, init = 'uniform', activation = 'sigmoid'))

/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:2: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="relu", input_dim=11, units=6, kernel_initializer="uniform")`
  
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:3: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="relu", units=6, kernel_initializer="uniform")`
  This is separate from the ipykernel package so we can avoid doing imports until
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="sigmoid", units=1, kernel_initializer="uniform")`
  after removing the cwd from sys.path.


In [0]:
model.compile(optimizer = 'adam', loss = "binary_crossentropy", metrics = ['accuracy'])

In [85]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,batch_size=10)

Train on 8000 samples, validate on 2000 samples
Epoch 1/100
8000/8000 [==============================] - 1s 159us/step - loss: 0.4005 - acc: 0.8361 - val_loss: 0.3958 - val_acc: 0.8400
Epoch 2/100
8000/8000 [==============================] - 1s 149us/step - loss: 0.4000 - acc: 0.8356 - val_loss: 0.3957 - val_acc: 0.8410
Epoch 3/100
8000/8000 [==============================] - 1s 152us/step - loss: 0.4005 - acc: 0.8365 - val_loss: 0.3976 - val_acc: 0.8430
Epoch 4/100
8000/8000 [==============================] - 1s 148us/step - loss: 0.4005 - acc: 0.8334 - val_loss: 0.3963 - val_acc: 0.8400
Epoch 5/100
8000/8000 [==============================] - 1s 151us/step - loss: 0.4001 - acc: 0.8352 - val_loss: 0.3963 - val_acc: 0.8430
Epoch 6/100
8000/8000 [==============================] - 1s 150us/step - loss: 0.3996 - acc: 0.8347 - val_loss: 0.3980 - val_acc: 0.8420
Epoch 7/100
8000/8000 [==============================] - 1s 150us/step - loss: 0.4005 - acc: 0.8349 - val_loss: 0.3945 - val_acc: 

In [0]:
y_pred = model.predict(X_test)

In [95]:
y_pred

array([[0.1539639 ],
       [0.31615403],
       [0.16924   ],
       ...,
       [0.17034832],
       [0.12575877],
       [0.09079087]], dtype=float32)

In [0]:
# use Threshold of 0.5
y_pred_new = []
for val in y_pred:
    if val > 0.5: 
        y_pred_new.append(1)
    else:
        y_pred_new.append(0)

In [103]:
y_pred_new

[0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [0]:
from sklearn import metrics
from sklearn.metrics import confusion_matrix, accuracy_score


In [109]:
accuracy_score(y_test,y_pred_new) * 100

84.2

In [105]:
confusion_matrix(y_test, y_pred_new)

array([[1551,   44],
       [ 272,  133]])

In [0]:
report = metrics.classification_report(y_test,y_pred_new)


In [117]:
print(report)

              precision    recall  f1-score   support

           0       0.85      0.97      0.91      1595
           1       0.75      0.33      0.46       405

    accuracy                           0.84      2000
   macro avg       0.80      0.65      0.68      2000
weighted avg       0.83      0.84      0.82      2000



In [0]:
# Trying to find best activation and optimizer for the model. 

In [0]:
dic = {'relu':['sgd','adam'],'tanh':['sgd','adam'],'sigmoid':['sgd','adam']}

In [125]:
dict.keys()

dict_keys(['relu', 'tanh', 'sigmoid'])

In [0]:
for k in dic:
  print (k)
  model = Sequential()
  model.add(Dense(output_dim = 6, init = 'uniform', activation = k, input_dim = 11))
  model.add(Dense(output_dim = 6, init = 'uniform', activation = 'relu'))
  model.add(Dense(output_dim = 1, init = 'uniform', activation = 'sigmoid'))
  for opt in dic[k]:
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=10)
        
  

relu


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="relu", input_dim=11, units=6, kernel_initializer="uniform")`
  after removing the cwd from sys.path.
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:5: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="relu", units=6, kernel_initializer="uniform")`
  """
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:6: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="sigmoid", units=1, kernel_initializer="uniform")`
  


Train on 8000 samples, validate on 2000 samples
Epoch 1/10
8000/8000 [==============================] - 3s 336us/step - loss: 0.5576 - acc: 0.7951 - val_loss: 0.5113 - val_acc: 0.7975
Epoch 2/10
8000/8000 [==============================] - 1s 157us/step - loss: 0.5083 - acc: 0.7960 - val_loss: 0.5045 - val_acc: 0.7975
Epoch 3/10
8000/8000 [==============================] - 1s 161us/step - loss: 0.5061 - acc: 0.7960 - val_loss: 0.5039 - val_acc: 0.7975
Epoch 4/10
8000/8000 [==============================] - 1s 160us/step - loss: 0.5060 - acc: 0.7960 - val_loss: 0.5039 - val_acc: 0.7975
Epoch 5/10
8000/8000 [==============================] - 1s 158us/step - loss: 0.5060 - acc: 0.7960 - val_loss: 0.5039 - val_acc: 0.7975
Epoch 6/10
8000/8000 [==============================] - 1s 157us/step - loss: 0.5060 - acc: 0.7960 - val_loss: 0.5038 - val_acc: 0.7975
Epoch 7/10
8000/8000 [==============================] - 1s 161us/step - loss: 0.5060 - acc: 0.7960 - val_loss: 0.5038 - val_acc: 0.7975


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="tanh", input_dim=11, units=6, kernel_initializer="uniform")`
  after removing the cwd from sys.path.


Train on 8000 samples, validate on 2000 samples
Epoch 1/10
8000/8000 [==============================] - 3s 332us/step - loss: 0.5579 - acc: 0.7956 - val_loss: 0.5116 - val_acc: 0.7975
Epoch 2/10
8000/8000 [==============================] - 1s 162us/step - loss: 0.5085 - acc: 0.7960 - val_loss: 0.5045 - val_acc: 0.7975
Epoch 3/10
8000/8000 [==============================] - 1s 162us/step - loss: 0.5061 - acc: 0.7960 - val_loss: 0.5039 - val_acc: 0.7975
Epoch 4/10
8000/8000 [==============================] - 1s 160us/step - loss: 0.5059 - acc: 0.7960 - val_loss: 0.5038 - val_acc: 0.7975
Epoch 5/10
8000/8000 [==============================] - 1s 161us/step - loss: 0.5059 - acc: 0.7960 - val_loss: 0.5038 - val_acc: 0.7975
Epoch 6/10
8000/8000 [==============================] - 1s 161us/step - loss: 0.5059 - acc: 0.7960 - val_loss: 0.5037 - val_acc: 0.7975
Epoch 7/10
8000/8000 [==============================] - 1s 162us/step - loss: 0.5059 - acc: 0.7960 - val_loss: 0.5037 - val_acc: 0.7975


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:4: UserWarning: Update your `Dense` call to the Keras 2 API: `Dense(activation="sigmoid", input_dim=11, units=6, kernel_initializer="uniform")`
  after removing the cwd from sys.path.


Train on 8000 samples, validate on 2000 samples
Epoch 1/10
8000/8000 [==============================] - 3s 348us/step - loss: 0.5577 - acc: 0.7960 - val_loss: 0.5111 - val_acc: 0.7975
Epoch 2/10
8000/8000 [==============================] - 1s 163us/step - loss: 0.5082 - acc: 0.7960 - val_loss: 0.5043 - val_acc: 0.7975
Epoch 3/10
8000/8000 [==============================] - 1s 163us/step - loss: 0.5060 - acc: 0.7960 - val_loss: 0.5038 - val_acc: 0.7975
Epoch 4/10
8000/8000 [==============================] - 1s 163us/step - loss: 0.5058 - acc: 0.7960 - val_loss: 0.5039 - val_acc: 0.7975
Epoch 5/10
8000/8000 [==============================] - 1s 162us/step - loss: 0.5058 - acc: 0.7960 - val_loss: 0.5036 - val_acc: 0.7975
Epoch 6/10
8000/8000 [==============================] - 1s 165us/step - loss: 0.5058 - acc: 0.7960 - val_loss: 0.5036 - val_acc: 0.7975
Epoch 7/10
8000/8000 [==============================] - 1s 168us/step - loss: 0.5057 - acc: 0.7960 - val_loss: 0.5035 - val_acc: 0.7975
